In [11]:
# ==========================================================
# Global Random Forest for Customer Forecasting
# ==========================================================

import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Nixtla
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from mlforecast.target_transforms import Differences, LocalBoxCox

# Machine Learning
from sklearn.ensemble import RandomForestRegressor

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# ==========================================================
# Load Data
# ==========================================================

processed_dir = Path("data/processed")

sales = pd.read_csv(
    processed_dir / "sales_clean.csv",
    parse_dates=["date"],
    low_memory=False
)

future = pd.read_csv(
    processed_dir / "future_clean.csv",
    parse_dates=["date"],
    low_memory=False
)

# ==========================================================
# Remove Unavailable Feature
# ==========================================================
# Sales are unavailable when forecasting customers.

sales = sales.drop(columns=["sales"])

# ==========================================================
# Sort Data
# ==========================================================

sales = sales.sort_values(
    ["store_id", "date"]
).reset_index(drop=True)

future = future.sort_values(
    ["store_id", "date"]
).reset_index(drop=True)

# ==========================================================
# Rename Columns for MLForecast
# ==========================================================

sales = sales.rename(
    columns={
        "store_id": "unique_id",
        "date": "ds",
        "customers": "y"
    }
)

future = future.rename(
    columns={
        "store_id": "unique_id",
        "date": "ds"
    }
)

# ==========================================================
# Remove Target from Future Data
# ==========================================================

future = future.drop(columns=["customers"])

# ==========================================================
# Encode Categorical Features
# ==========================================================

state_holiday_map = {
    "0": 0,
    "a": 1,
    "b": 2,
    "c": 3,
    0: 0,
    0.0: 0
}

store_type_map = {
    "a": 0,
    "b": 1,
    "c": 2,
    "d": 3
}

assortment_map = {
    "a": 0,
    "b": 1,
    "c": 2
}

for df in [sales, future]:
    df["state_holiday"] = (
        df["state_holiday"]
        .replace(state_holiday_map)
        .astype(int)
    )

    df["store_type"] = (
        df["store_type"]
        .map(store_type_map)
        .astype(int)
    )

    df["assortment"] = (
        df["assortment"]
        .map(assortment_map)
        .astype(int)
    )

# ==========================================================
# Create Additional Dynamic Features
# ==========================================================

# Weekend indicator
sales["is_weekend"] = (
        sales["ds"].dt.dayofweek >= 5
).astype(int)

future["is_weekend"] = (
        future["ds"].dt.dayofweek >= 5
).astype(int)

# Quarter
sales["quarter"] = sales["ds"].dt.quarter
future["quarter"] = future["ds"].dt.quarter

# ISO week number
sales["week_of_year"] = (
    sales["ds"]
    .dt.isocalendar()
    .week
    .astype(int)
)

future["week_of_year"] = (
    future["ds"]
    .dt.isocalendar()
    .week
    .astype(int)
)

# Promotion during weekends
sales["promo_weekend"] = (
        sales["promo"] *
        sales["is_weekend"]
)

future["promo_weekend"] = (
        future["promo"] *
        future["is_weekend"]
)

# ==========================================================
# Check Data
# ==========================================================

print("=" * 60)
print("Training Data")
print("=" * 60)

print(sales.head())

print()

print(sales.dtypes)

print()

print("=" * 60)
print("Future Data")
print("=" * 60)

print(future.head())

print()

print(future.dtypes)

print()

print("=" * 60)
print("Training Columns")
print("=" * 60)

print(sales.columns.tolist())

print()

print("=" * 60)
print("Future Columns")
print("=" * 60)

print(future.columns.tolist())

Training Data
  unique_id         ds    y  open  promo  state_holiday  school_holiday  \
0   store_1 2013-01-07  785     1      1              0               1   
1   store_1 2013-01-08  654     1      1              0               1   
2   store_1 2013-01-09  626     1      1              0               1   
3   store_1 2013-01-10  615     1      1              0               1   
4   store_1 2013-01-11  592     1      1              0               1   

   store_type  assortment  competition_distance  is_weekend  quarter  \
0           2           0                1270.0           0        1   
1           2           0                1270.0           0        1   
2           2           0                1270.0           0        1   
3           2           0                1270.0           0        1   
4           2           0                1270.0           0        1   

   week_of_year  promo_weekend  
0             2              0  
1             2              0  
2  

In [12]:


# ==========================================================
# Hyperparameter Grid
# ==========================================================

parameter_grid = [

    {
        "name": "RF200_SQRT",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },

    {
        "name": "RF300_SQRT_Leaf5",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": "sqrt"
    }

]

# ==========================================================
# Hyperparameter Tuning with Rolling Cross Validation
# ==========================================================

results = []

for params in parameter_grid:
    print("=" * 60)
    print(params["name"])
    print("=" * 60)

    model = RandomForestRegressor(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        max_features=params["max_features"],
        random_state=42,
        n_jobs=-1,

)

    fcst = MLForecast(

        models=[model],

        freq="D",

        lags=[
            1,
            7,
            14
        ],

        lag_transforms={
            7: [
                RollingMean(window_size=4)
            ]
        },

        date_features=[
            "dayofweek",
            "month"
        ]

    )

    cv = fcst.cross_validation(

        df=sales,

        h=42,

        n_windows=3,

        refit=False,

        static_features=[]

    )

    prediction_col = cv.columns[-1]

    valid = cv["y"] > 0

    mae = mean_absolute_error(
        cv.loc[valid, "y"],
        cv.loc[valid, prediction_col]
    )

    rmse = np.sqrt(
        mean_squared_error(
            cv.loc[valid, "y"],
            cv.loc[valid, prediction_col]
        )
    )

    mape = (
            np.abs(
                (
                        cv.loc[valid, "y"]
                        - cv.loc[valid, prediction_col]
                )
                /
                cv.loc[valid, "y"]
            ).mean()
            * 100
    )

    results.append({

        "Model": params["name"],

        "Trees": params["n_estimators"],

        "MAE": mae,

        "RMSE": rmse,

        "MAPE": mape

    })

    print(f"MAE : {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAPE: {mape:.2f}%")

# ==========================================================
# Save Cross Validation Results
# ==========================================================

results = pd.DataFrame(results)

results = results.sort_values(
    by="MAPE",
    ascending=True
)

print(results)

results.to_csv(
    processed_dir / "customer_rf_cv_results.csv",
    index=False
)

RF200_SQRT
MAE : 65.46
RMSE: 97.85
MAPE: 8.63%
RF300_SQRT_Leaf5
MAE : 66.51
RMSE: 101.20
MAPE: 8.77%
              Model  Trees        MAE        RMSE      MAPE
0        RF200_SQRT    200  65.461082   97.851761  8.625688
1  RF300_SQRT_Leaf5    300  66.514957  101.203625  8.767172


In [13]:
# ==========================================================
# Select Best Hyperparameters
# ==========================================================

best_result = results.loc[
    results["MAPE"].idxmin()
]

best_trees = int(best_result["Trees"])

print("=" * 60)
print("Best Hyperparameters")
print("=" * 60)

print(best_result)

# ==========================================================
# Keep First 42 Forecast Days
# ==========================================================

future_42 = (
    future
    .groupby("unique_id", group_keys=False)
    .head(42)
)

print(future_42.shape)

# ==========================================================
# Build Final Forecast Model
# ==========================================================

final_model = RandomForestRegressor(

    n_estimators=best_trees,

    min_samples_leaf=1,

    random_state=42,

    n_jobs=-1

)

final_fcst = MLForecast(

    models=[final_model],

    freq="D",

    lags=[
        1,
        7,
        14
    ],

    lag_transforms={
        7: [
            RollingMean(window_size=4)
        ]
    },

    date_features=[
        "dayofweek",
        "month"
    ]

)

# ==========================================================
# Train Final Model on Full Historical Data
# ==========================================================

final_fcst.fit(

    sales,

    static_features=[]

)

print("Final model trained.")

# ==========================================================
# Forecast Future Customers
# ==========================================================

customer_forecast = final_fcst.predict(

    h=42,

    X_df=future_42

)


prediction_col = customer_forecast.columns[-1]

customer_forecast = customer_forecast.rename(

    columns={
        prediction_col: "customer_prediction"
    }

)

print("=" * 60)
print("Customer Forecast")
print("=" * 60)

print(customer_forecast.head())

print(customer_forecast.shape)

# ==========================================================
# Merge Forecast with Future Data
# ==========================================================

future_customer = future_42.merge(

    customer_forecast,

    on=[
        "unique_id",
        "ds"
    ],

    how="left"

)

print("=" * 60)
print("Missing Predictions")
print("=" * 60)

print(
    future_customer["customer_prediction"]
    .isna()
    .sum()
)

print()

print(future_customer.head())

# ==========================================================
# Save Customer Forecast
# ==========================================================

customer_forecast = future_customer[
    [
        "unique_id",
        "ds",
        "customer_prediction"
    ]
].copy()

output_file = processed_dir / "future_customer_prediction_rf.csv"

customer_forecast.to_csv(

    output_file,

    index=False

)

print("=" * 60)
print("Customer forecast saved successfully.")
print("=" * 60)

print(customer_forecast.head())
print(output_file)

Best Hyperparameters
Model    RF200_SQRT
Trees           200
MAE       65.461082
RMSE      97.851761
MAPE       8.625688
Name: 0, dtype: object
(28392, 13)
Final model trained.
Customer Forecast
  unique_id         ds  customer_prediction
0   store_1 2015-07-20              486.060
1   store_1 2015-07-21              446.065
2   store_1 2015-07-22              461.360
3   store_1 2015-07-23              470.705
4   store_1 2015-07-24              462.035
(28392, 3)
Missing Predictions
0

  unique_id         ds  open  promo  state_holiday  school_holiday  \
0   store_1 2015-07-20   1.0      0              0               0   
1   store_1 2015-07-21   1.0      0              0               0   
2   store_1 2015-07-22   1.0      0              0               0   
3   store_1 2015-07-23   1.0      0              0               0   
4   store_1 2015-07-24   1.0      0              0               0   

   store_type  assortment  competition_distance  is_weekend  quarter  \
0           2 